# AMPS II · Week 15 — Project workshop
## A verification kit for your project: predicted vs observed rates, the right yardstick, a QoI without an exact solution, `-np 1` = `-np 2`, a run-time budget

| | |
|---|---|
| **Workshop slides** | Part B (the progress talk) · Part C (verification at mid-project) · Part D (the code review) |
| **Time** | ≈ 45 min, in the workshop block while the code reviews run; finish before your talk if you can |
| **You will** | predict and then measure convergence rates on a worked example · see how a wrong error measure produces convincing but false rates · estimate the error of a QoI that has no exact value (Richardson, GCI) · check that `-np 1` and `-np 2` agree · estimate the Colab run time of your final study · apply the same helpers to **your project** (§8) and make the verification slide of your progress talk |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. Two modules are written once with `%%writefile` (§1):
`projkit.py` (NumPy only — **copy it into your project notebook**; it works in ordinary cells and in `%%fenicsx` cells) and `rdx.py` (the worked example).

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The modules
**`projkit.py`** (NumPy only): `rates(h_or_N, errors)`, `rate_table(h_or_N, {name: errors}, {name: predicted})`, `richardson([q1, q2, q3], r=2)` → observed order,
extrapolated value, error estimate and GCI, `budget(unknowns, seconds, target)`, `record(tag, comm, **values)` (JSON per `-np`), `compare_np(tag)`, `rate_plot(...)`.

**`rdx.py`** — the worked example: steady nonlinear reaction–diffusion on the unit square,
$$-\nabla\cdot\big((1+u^2)\nabla u\big)+\mathrm{Da}\,u^2=f,\qquad u=g\ \text{on}\ \partial\Omega,$$
Newton (SNES) with MUMPS. `solve(N, deg, Da=1, case="mms" | "physical")`: with `"mms"` the data come from the manufactured solution $u=e^{x}\sin\pi y+\tfrac12y^2$;
with `"physical"` $f=10$, $g=0$ and there is no exact solution. The quantity of interest is the total reaction rate $Q=\int_\Omega\mathrm{Da}\,u^2\,dx$.
Returns the solution, SNES reason and iterations, DOFs, wall time, $Q$ and (MMS) the errors. Run both cells.

In [ ]:
%%writefile /tmp/fnx/projkit.py
"""projkit.py — verification helpers for the term project (AMPS II, Week 15). NumPy only: import it in ordinary cells and in %%fenicsx cells."""
import json, os
import numpy as np

def rates(h, e):
    """observed orders between successive meshes: log(e_i / e_i+1) / log(h_i / h_i+1).
    h may also be the cell counts N (an increasing sequence is read as N, h = 1/N)"""
    h, e = np.asarray(h, float), np.asarray(e, float)
    if h[-1] > h[0]:
        h = 1.0 / h
    return np.log(e[:-1] / e[1:]) / np.log(h[:-1] / h[1:])

def rate_table(h, errs, predicted=None, label="h"):
    """a text table: one column per error (errs = {name: [e_1, ..., e_n]}), the observed rates, and the predicted rate in the last row
    (h or N as in rates(); label is the name of the first column)"""
    h = np.asarray(h, float); predicted = predicted or {}
    head = f"{label:>10s} " + " ".join(f"{n:>11s} {'rate':>5s}" for n in errs)
    rows = [head, "-" * len(head)]
    R = {n: rates(h, e) for n, e in errs.items()}
    for i in range(len(h)):
        rows.append(f"{h[i]:10.4g} " + " ".join(f"{errs[n][i]:11.3e} " + (f"{R[n][i - 1]:5.2f}" if i else "     ") for n in errs))
    rows.append(f"{'predicted':>10s} " + " ".join(f"{'':11s} " + (f"{predicted[n]:5.2f}" if n in predicted else "    ?") for n in errs))
    return "\n".join(rows)

def richardson(q, r=2.0):
    """QoI on three meshes, coarse to fine, refinement ratio r. Returns the observed order p, the extrapolated value,
    the estimated error of the finest value, and the grid-convergence index GCI = 1.25 |q3 - q2| / (r^p - 1) / |q3|.
    Monotone convergence is required: if the differences change sign, p is meaningless (returned as nan)."""
    q1, q2, q3 = map(float, q[-3:])
    d1, d2 = q1 - q2, q2 - q3
    if d1 * d2 <= 0:
        return dict(p=np.nan, q_ext=np.nan, err=np.nan, gci=np.nan, monotone=False)
    p = np.log(d1 / d2) / np.log(r)
    q_ext = q3 - d2 / (r ** p - 1)
    return dict(p=p, q_ext=q_ext, err=abs(q3 - q_ext), gci=1.25 * abs(d2) / (r ** p - 1) / abs(q3), monotone=True)

def budget(n, t, n_target):
    """fit t = a n^p to the last two points (n = unknowns, t = seconds) and predict the time at n_target"""
    n, t = np.asarray(n, float), np.asarray(t, float)
    p = np.log(t[-1] / t[-2]) / np.log(n[-1] / n[-2])
    return dict(p=p, t=t[-1] * (n_target / n[-1]) ** p)

def record(tag, comm, **values):
    """write the QoI of this run to /tmp/fnx/qoi_<tag>_np<size>.json (rank 0); comm is the MPI communicator of the %%fenicsx cell"""
    if comm.rank == 0:
        with open(f"/tmp/fnx/qoi_{tag}_np{comm.size}.json", "w") as f:
            json.dump({k: float(v) for k, v in values.items()}, f)

def compare_np(tag, a=1, b=2, rtol=1e-8):
    """compare the QoI written by record() from -np a and -np b runs; returns True if all agree to rtol"""
    fa, fb = f"/tmp/fnx/qoi_{tag}_np{a}.json", f"/tmp/fnx/qoi_{tag}_np{b}.json"
    missing = [f for f in (fa, fb) if not os.path.exists(f)]
    if missing:
        print("missing:", ", ".join(missing), "- run the cell with both -np values first"); return False
    A, B = json.load(open(fa)), json.load(open(fb)); ok = True
    for k in A:
        d = abs(A[k] - B[k]) / max(abs(A[k]), 1e-300); ok &= d <= rtol
        print(f"{k:>12s}: np{a} {A[k]:.12g}  np{b} {B[k]:.12g}  relative difference {d:.1e}  {'ok' if d <= rtol else 'DIFFERENT'}")
    return bool(ok)

def rate_plot(h, errs, predicted=None, fname=None, xlabel="h", title=None):
    """log-log plot of the errors with a dashed line of the predicted slope (drawn 1.6 times above the finest point) for each predicted rate; saved to fname if given"""
    import matplotlib.pyplot as plt
    h = np.asarray(h, float); predicted = predicted or {}
    if h[-1] > h[0]:
        h = 1.0 / h
    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    for i, (n, e) in enumerate(errs.items()):
        e = np.asarray(e, float); col = f"C{i}"
        ax.loglog(h, e, "o-", color=col, label=f"{n} (observed {rates(h, e)[-1]:.2f})")
        if n in predicted:
            ax.loglog(h, 1.6 * e[-1] * (h / h[-1]) ** predicted[n], "--", color=col, lw=1, alpha=0.7, label=f"predicted slope {predicted[n]:g}")
    ax.set_xticks(h, [f"1/{1 / v:.0f}" if abs(1 / v - round(1 / v)) < 1e-9 else f"{v:.3g}" for v in h]); ax.set_xticks([], minor=True)
    ax.set_xlabel(xlabel); ax.set_ylabel("error"); ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8)
    if title:
        ax.set_title(title, fontsize=10)
    fig.tight_layout()
    if fname:
        fig.savefig(fname, dpi=150)
    return fig

In [ ]:
%%writefile /tmp/fnx/rdx.py
"""rdx.py — worked example for the progress check (AMPS II, Week 15), DOLFINx 0.11:
steady nonlinear reaction-diffusion  -div((1 + u^2) grad u) + Da u^2 = f  on the unit square, u = g on the boundary, Newton (SNES) + MUMPS."""
import gc, time
import numpy as np
import ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem

comm = MPI.COMM_WORLD
gc.disable()                                    # PETSc destructors are collective: collect at the same point on all ranks
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
def total(form):
    return comm.allreduce(fem.assemble_scalar(fem.form(form)), op=MPI.SUM)

def u_exact(x):
    """the manufactured solution, as a UFL expression of the coordinates (or of a NumPy array of points)"""
    m = ufl if isinstance(x, ufl.core.expr.Expr) else np
    return m.exp(x[0]) * m.sin(np.pi * x[1]) + 0.5 * x[1] ** 2

_count = [0]
def solve(N, deg=1, Da=1.0, case="mms", f_value=10.0, cell="triangle", rtol=1e-10):
    """case "mms": f and g from u_exact; case "physical": f = f_value, g = 0 (no exact solution).
    Returns the solution, the mesh, SNES reason and iterations, wall time, DOFs, the QoI Q = int Da u^2 dx, and (mms) the errors:
    L2 and H1-seminorm error against the exact expression (quadrature raised), and the same two against the interpolant of u_exact in V."""
    gc.collect(); _count[0] += 1; t0 = time.perf_counter()
    ct = mesh.CellType.triangle if cell == "triangle" else mesh.CellType.quadrilateral
    msh = mesh.create_unit_square(comm, N, N, ct)
    V = fem.functionspace(msh, ("Lagrange", deg))
    x = ufl.SpatialCoordinate(msh); u, v = fem.Function(V), ufl.TestFunction(V)
    k = lambda w: 1 + w ** 2
    if case == "mms":
        ue = u_exact(x); f = -ufl.div(k(ue) * ufl.grad(ue)) + Da * ue ** 2
        g = fem.Function(V); g.interpolate(lambda X: u_exact(X))
    else:
        f = fem.Constant(msh, f_value); g = fem.Function(V)
    fdim = msh.topology.dim - 1; msh.topology.create_connectivity(fdim, msh.topology.dim)
    bc = fem.dirichletbc(g, fem.locate_dofs_topological(V, fdim, mesh.exterior_facet_indices(msh.topology)))
    dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": 2 * deg + 6})
    F = (k(u) * ufl.dot(ufl.grad(u), ufl.grad(v)) + Da * u ** 2 * v - f * v) * dx
    opts = {"snes_type": "newtonls", "snes_linesearch_type": "basic", "snes_rtol": rtol, "snes_atol": 1e-12, "snes_max_it": 30,
            "ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}
    problem = NonlinearProblem(F, u, bcs=[bc], petsc_options_prefix=f"rdx{_count[0]}_", petsc_options=opts)
    problem.solve()
    snes = problem.solver
    out = dict(u=u, msh=msh, N=N, h=1.0 / N, deg=deg, reason=snes.getConvergedReason(), its=snes.getIterationNumber(),
               dofs=V.dofmap.index_map.size_global, time=time.perf_counter() - t0, Q=total(Da * u ** 2 * dx))
    if case == "mms":
        e = u - ue
        out["L2"] = np.sqrt(total(e ** 2 * dx)); out["H1"] = np.sqrt(total(ufl.dot(ufl.grad(e), ufl.grad(e)) * dx))
        Iu = fem.Function(V); Iu.interpolate(lambda X: u_exact(X)); ei = u - Iu
        out["L2_interp"] = np.sqrt(total(ei ** 2 * dx)); out["H1_interp"] = np.sqrt(total(ufl.dot(ufl.grad(ei), ufl.grad(ei)) * dx))
        out["Q_exact"] = total(Da * ue ** 2 * dx)
    return out

## 2 · Predict, then measure  *(slides §C1)*
A convergence study is a **test only if you write the predicted rate down first.** "The error decreases" is not a test; "the $L^2$ error of P2 decreases like $h^3$, and it does" is.

> **✏️ Your turn (3 min) — before running the next cell.** For P1 and P2 on this smooth solution, predict the rates of (i) the $L^2$ error, (ii) the $H^1$-seminorm error, (iii) the error of the QoI $Q$.

> *your predictions*

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from rdx import *
from projkit import *

import json
solve(4, 1); solve(4, 2)                                  # JIT warm-up (the first call of a new form compiles it)
NS = [8, 16, 32, 64]; out = {}
for deg in (1, 2):
    R = [solve(N, deg) for N in NS]
    say(f"P{deg}: SNES reasons {[r['reason'] for r in R]}, Newton its {[r['its'] for r in R]}, dofs {[r['dofs'] for r in R]}")
    E = {k: [float(r[k]) for r in R] for k in ("L2", "H1", "L2_interp", "H1_interp")}
    E["Q"] = [abs(float(r["Q"] - r["Q_exact"])) for r in R]
    say(rate_table(NS, E, {"L2": deg + 1, "H1": deg, "Q": 2 * deg}, label="N"))
    out[deg] = E
if comm.rank == 0:
    json.dump(out, open("/tmp/fnx/w15_mms.json", "w"))

The figure for your progress talk: errors on log–log axes with the **predicted** slopes as dashed lines and the observed rates in the legend (`rate_plot`). Saved to `/tmp/fnx/rates_P2.png`.

In [ ]:
import sys, json; sys.path.insert(0, "/tmp/fnx")
import numpy as np, matplotlib.pyplot as plt
from projkit import rate_plot
E = json.load(open("/tmp/fnx/w15_mms.json"))["2"]
rate_plot([8, 16, 32, 64], {"L2": E["L2"], "H1": E["H1"], "Q": E["Q"]}, {"L2": 3, "H1": 2, "Q": 4}, fname="/tmp/fnx/rates_P2.png", title="P2, manufactured solution")
plt.show()

## 3 · The wrong yardstick  *(slides §C2)*
The columns `L2_interp` and `H1_interp` measure $u_h-I_hu$: the distance between the discrete solution and the **interpolant** of the exact solution in the same space — a very common way to "compute the error"
(`u_ex = fem.Function(V); u_ex.interpolate(...)`, then the norm of the difference). The correct columns integrate $u_h-u$ with the exact **expression** and a raised quadrature degree.

> **✏️ Your turn (5 min).** (i) Compare `H1` and `H1_interp` for P1: by what factor do they differ at $N=64$, and which rate does each show? (ii) Why is $u_h-I_hu$ not the discretisation error, and why is a **better-than-predicted** rate also a failed test?

> *your answer*

## 4 · The QoI converges at its own rate  *(slides §C1)*
Look again at the `Q` column of §2: P1 rate 2.00, P2 rate 4.02 — the predicted $2p$. Report the rate of the **quantity you report**, not only of the field: a QoI that involves a derivative (a flux, a Nusselt
number, a reaction force from the gradient) usually converges **slower** than $L^2$ — at the $H^1$ rate $p$ — unless you compute it from the residual (consistent flux, Weeks 1 and 11).

## 5 · No exact solution: Richardson extrapolation and the GCI  *(slides §C3)*
Your production run has no exact solution. With the QoI on three meshes $h,\ h/2,\ h/4$ (ratio $r=2$) and **monotone** convergence,
$$p_\text{obs}=\frac{\ln\big((Q_1-Q_2)/(Q_2-Q_3)\big)}{\ln r},\qquad Q_\infty\approx Q_3+\frac{Q_3-Q_2}{r^{p}-1},\qquad \mathrm{GCI}=1.25\,\frac{|Q_3-Q_2|}{(r^{p}-1)\,|Q_3|}.$$
The test is that $p_\text{obs}$ is close to the **predicted** rate; then $|Q_3-Q_\infty|$ estimates the error of your finest value, and the GCI (with a safety factor 1.25) is the uncertainty to report.
The cell computes the physical case with P1 on five meshes and an independent check: P2 on $N=256$.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from rdx import *
from projkit import *

NP = [8, 16, 32, 64, 128]
P = [solve(N, 1, case="physical") for N in NP]
say("SNES reasons", [r["reason"] for r in P], "| Newton its", [r["its"] for r in P])
Q = [float(r["Q"]) for r in P]
say("Q:", " ".join(f"{q:.7f}" for q in Q))
ref = solve(256, 2, case="physical"); Qref = float(ref["Q"])
say(f"independent check: P2, N = 256: Q = {Qref:.10f} (reason {ref['reason']}, {ref['dofs']} dofs)")
for i in range(3):
    r = richardson(Q[i:i + 3])
    say(f"meshes {NP[i]:3d}-{NP[i + 1]:3d}-{NP[i + 2]:3d}: p_obs {r['p']:.3f}, Q_ext {r['q_ext']:.7f}, estimated error {r['err']:.2e}, "
        f"actual error of Q({NP[i + 2]}) {abs(Q[i + 2] - Qref):.2e}, GCI {100 * r['gci']:.3f} %")

> **✏️ Your turn (5 min).** (i) How good is the Richardson error estimate from the three **coarsest** meshes? (ii) What would you write in your report for $Q$ from the meshes 32–64–128? (iii) What does it mean if the differences $Q_1-Q_2$ and $Q_2-Q_3$ have
> opposite signs, or if $p_\text{obs}$ is 0.8 where you predicted 2?

> *your answer*

## 6 · `-np 1` = `-np 2`  *(slides §C4)*
Run the **same** cell with one and with two processes; each run writes its QoI with `record(tag, comm, ...)`, and an ordinary cell compares them. Values must agree to round-off (a relative difference of $10^{-12}$, not $10^{-4}$).
Different values mean a partition-dependent bug: random numbers drawn per rank, a sum that is not reduced over ranks, a point value read from a ghost cell, a quantity counted twice on shared facets.

In [ ]:
%%fenicsx -np 1
import sys; sys.path.insert(0, "/tmp/fnx")
from rdx import *
from projkit import *

r = solve(64, 1, case="physical")
record("rdx64", comm, Q=r["Q"], dofs=r["dofs"], newton_its=r["its"])
say(f"-np {comm.size}: Q = {r['Q']:.12f}, reason {r['reason']}, its {r['its']}")

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from rdx import *
from projkit import *

r = solve(64, 1, case="physical")
record("rdx64", comm, Q=r["Q"], dofs=r["dofs"], newton_its=r["its"])
say(f"-np {comm.size}: Q = {r['Q']:.12f}, reason {r['reason']}, its {r['its']}")

In [ ]:
import sys; sys.path.insert(0, "/tmp/fnx")
from projkit import compare_np
compare_np("rdx64", rtol=1e-10)

## 7 · A run-time budget  *(slides §C5)*
The final notebook must run on Colab in **at most 30 minutes** (Week 16 guide). Measure the cost of one solve on a few meshes, **after** a warm-up call (the first call of a new form includes the JIT compilation), take the minimum of repeats,
fit $t\propto n^{p}$ and extrapolate to the size of your final study.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from rdx import *
from projkit import *

import time
a = time.perf_counter(); solve(16, 3); first = time.perf_counter() - a
a = time.perf_counter(); solve(16, 3); second = time.perf_counter() - a
say(f"P3, N = 16: first call {first:.2f} s (includes JIT), second call {second:.3f} s")
rows = []
for N in (32, 64, 128, 256):
    ts = [comm.allreduce(solve(N, 2)["time"], op=MPI.MAX) for rep in range(3)]
    r = solve(N, 2); rows.append([r["dofs"], min(ts)])
    say(f"P2, N = {N:3d}: {r['dofs']:6d} dofs, {min(ts):.3f} s (min of 3), Newton its {r['its']}")
b = budget([x[0] for x in rows], [x[1] for x in rows], 4 * rows[-1][0])
say(f"t ~ n^{b['p']:.2f}; predicted at {4 * rows[-1][0]} dofs: {b['t']:.1f} s per solve")

> **✏️ Your turn (4 min).** (i) Why must the warm-up call be excluded? (ii) Your final study needs 6 parameter values × 4 meshes, the finest with 4 times the unknowns of your current finest mesh, and Colab is about 4 times slower than your
> measurement machine. Write the budget as a formula, and name two ways to stay below 30 minutes without losing the verification.

> *your answer*

## 8 · Your project — the verification evidence for the progress talk
Copy the two `projkit` lines below into your `project.ipynb` and use them on **your** solver. What the progress talk needs (slides §B): one convergence plot with predicted slopes (manufactured or exact solution, at least three meshes),
the status of your topic's physical check, and `-np 1` vs `-np 2` for your QoI if you have it.

```python
# in a %%fenicsx cell of your project (after the %%writefile cell of projkit.py)
import sys; sys.path.insert(0, "/tmp/fnx"); from projkit import *
errs = [...]                                          # your errors on N = 8, 16, 32, 64 (u - u_h with the exact expression)
say(rate_table([8, 16, 32, 64], {"L2": errs}, {"L2": 2}, label="N"))   # predicted: YOUR prediction
record("myqoi", comm, Q=my_qoi)                       # then the same cell with -np 1, and compare_np("myqoi") in an ordinary cell
```

**Status table for slide 2 of the talk** — fill it in from your proposal's milestones:

| Milestone (from the proposal) | Status (done / partial / not started) | Evidence (cell, number, figure) |
|---|---|---|
| | | |
| | | |
| | | |

In [ ]:
# (your project) — paste projkit.py into your project notebook and make your verification figure here or there

## 9 · Before your talk and your code review
- [ ] `project.ipynb`: *Runtime → Restart session and run all* finishes; every `%%fenicsx` output read (an error there does not stop *Run all*)
- [ ] every solve that a number depends on prints its converged reason (SNES/KSP), and the reason is positive
- [ ] the errors are $u-u_h$ with the exact expression (not against the interpolant); the predicted rates are written **before** the table
- [ ] one convergence figure with predicted slopes (§2 style) for slide 3
- [ ] your QoI on three meshes with $p_\text{obs}$, or a sentence why it is not yet monotone
- [ ] `-np 1` and `-np 2` agree for your QoI (or you know why not)
- [ ] the run-time budget of your final study (§7)
- [ ] the AI-usage log of your project is up to date — the code review asks about it

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …